In [17]:
import pandas as pd
import os

In [7]:
df = pd.read_csv("sensores_industriales.csv")

In [8]:
df.head()

,id_registro,fecha_hora,id_sensor,planta,temperatura_c,vibracion_mm_s
0,1,01/09/26 0:00,S001,Planta_1,65.93,1.40
1,2,01/09/26 0:00,S002,Planta_1,54.05,4.21
2,3,01/09/26 0:00,S003,Planta_1,66.34,2.26
3,4,01/09/26 0:00,S004,Planta_1,65.24,1.32
4,5,01/09/26 0:00,S005,Planta_1,68.12,3.22


In [9]:
df.tail()

,id_registro,fecha_hora,id_sensor,planta,temperatura_c,vibracion_mm_s
99995,99996,02/09/26 17:39,S036,Planta_4,81.32,2.70
99996,99997,02/09/26 17:39,S037,Planta_4,52.34,5.40
99997,99998,02/09/26 17:39,S038,Planta_4,52.18,2.30
99998,99999,02/09/26 17:39,S039,Planta_4,82.58,1.03
99999,100000,02/09/26 17:39,S040,Planta_4,82.40,4.44


Mostrar la cantidad de registros y de sensores distintos:

In [10]:
total_registros = len(df)
sensores_unicos = df['id_sensor'].nunique()
print(
    f"Total de registros: {total_registros:,} | Sensores distintos:"
    f" {sensores_unicos}"
)

Total de registros: 100,000 | Sensores distintos: 40


Calcular la temperatura promedio de cada planta

In [11]:
temp_prom_planta = df.groupby('planta')['temperatura_c'].mean()
print("\nTemperatura promedio por planta:\n", temp_prom_planta)


Temperatura promedio por planta:
 planta
Planta_1    66.616297
Planta_2    66.534989
Planta_3    66.765774
Planta_4    66.668966
Name: temperatura_c, dtype: float64


Encontrar la temperatura máxima e identificar el sensor y la fecha correspondientes

In [13]:
idx_max_temp = df['temperatura_c'].idxmax()
registro_max = df.loc[idx_max_temp]
print(f"\nTemperatura Máxima: {registro_max['temperatura_c']} °C")
print(f"Sensor: {registro_max['id_sensor']}")
print(f"Fecha y Hora: {registro_max['fecha_hora']}")


Temperatura Máxima: 104.99 °C
Sensor: S023
Fecha y Hora: 01/09/26 22:23


Contar las lecturas con temperatura mayor que 85 °C

In [14]:
df_alertas = df[df['temperatura_c'] > 85]
total_alertas = len(df_alertas)
print(f"\nTotal de lecturas con alerta (> 85 °C): {total_alertas}")


Total de lecturas con alerta (> 85 °C): 6954


Identificar la planta con más alertas de temperatura

In [15]:
planta_mas_alertas = df_alertas['planta'].value_counts().idxmax()
conteo_alertas_planta = df_alertas['planta'].value_counts().max()
print(
    f"\nPlanta con más alertas: {planta_mas_alertas} ({conteo_alertas_planta}"
    " alertas)"
)


Planta con más alertas: Planta_3 (1777 alertas)


Exportar todas las lecturas con alerta a resultados/alertas.csv
, conservando las columnas originales

In [18]:
os.makedirs("resultados", exist_ok=True)
df_alertas.to_csv("resultados/alertas.csv", index=False)
print("\nArchivo de alertas exportado exitosamente a 'resultados/alertas.csv'")


Archivo de alertas exportado exitosamente a 'resultados/alertas.csv'


In [19]:
%%writefile analisis.py
import os
import pandas as pd

ruta_csv = os.path.join("data", "sensores_industriales.csv")
if not os.path.exists(ruta_csv):
    ruta_csv = "sensores_industriales.csv"

df = pd.read_csv(ruta_csv)

# 1. Registros y sensores
print(f"Total registros: {len(df)} | Sensores: {df['id_sensor'].nunique()}")

# 2. Promedio por planta
print("\nTemperatura promedio por planta:\n", df.groupby('planta')['temperatura_c'].mean())

# 3. Temperatura máxima con empates
max_temp = df['temperatura_c'].max()
print(f"\nTemperatura máxima: {max_temp} °C")
print(df[df['temperatura_c'] == max_temp][['id_sensor', 'fecha_hora', 'planta']])

# 4. Alertas > 85 °C
df_alertas = df[df['temperatura_c'] > 85]
print(f"\nTotal alertas (> 85 °C): {len(df_alertas)}")

# 5. Planta con más alertas con empates
conteo = df_alertas['planta'].value_counts()
max_alertas = conteo.max()
plantas_max = conteo[conteo == max_alertas]
print(f"\nPlanta(s) con más alertas ({max_alertas}):\n", plantas_max)

# 6. Exportar usando ruta relativa
os.makedirs("resultados", exist_ok=True)
df_alertas.to_csv(os.path.join("resultados", "alertas.csv"), index=False)
print("\nAlertas exportadas a resultados/alertas.csv")

Writing analisis.py


In [21]:
!python3 analisis.py

Total registros: 100000 | Sensores: 40

Temperatura promedio por planta:
 planta
Planta_1    66.616297
Planta_2    66.534989
Planta_3    66.765774
Planta_4    66.668966
Name: temperatura_c, dtype: float64

Temperatura máxima: 104.99 °C
      id_sensor      fecha_hora    planta
53742      S023  01/09/26 22:23  Planta_3
89258      S019  02/09/26 13:11  Planta_2
94533      S014  02/09/26 15:23  Planta_2
96109      S030  02/09/26 16:02  Planta_3

Total alertas (> 85 °C): 6954

Planta(s) con más alertas (1777):
 planta
Planta_3    1777
Name: count, dtype: int64

Alertas exportadas a resultados/alertas.csv
